### 1. Duck Typing (Informal Interfaces)

In [ ]:


class PDFReader:
    def extract_text(self) -> str:
        return "Text from PDF"

class EmailReader:
    def extract_text(self) -> str:
        return "Text from Email"

# This function accepts any object as long as it has an `extract_text` method
def print_content(reader):
    print(reader.extract_text())

print_content(PDFReader())   # Works
print_content(EmailReader()) # Works


Text from PDF
Text from Email


### 2. What is an abstract method and Abstract class?

- https://www.w3schools.com/python/ref_module_abc.asp

In [3]:
from abc import ABC, abstractmethod

class FileReader(ABC):
    @abstractmethod
    def extract_text(self) -> str:
        pass

class PDFReader(FileReader):
    def extract_text(self) -> str:
        return "Text from PDF"

# This will fail at runtime because it didn't implement `extract_text`:
class FaultyReader(FileReader):
    pass

reader = FaultyReader() # Raises TypeError: Can't instantiate abstract class...


TypeError: Can't instantiate abstract class FaultyReader without an implementation for abstract method 'extract_text'

### 3. What if i dont want to use `@abstractmethod`

In [1]:
from abc import ABC, abstractmethod

class FileReader(ABC):
    def extract_text(self) -> str:
        pass

class PDFReader(FileReader):
    def extract_text(self) -> str:
        return "Text from PDF"

# This will fail at runtime because it didn't implement `extract_text`:
class FaultyReader(FileReader):
    pass

reader = FaultyReader() # Raises TypeError: Can't instantiate abstract class...


- it will not be an abstract method.


### 4.1 What is cls and classmethod in python?


In [8]:
class Pizza:
    # A class variable shared by all pizzas
    base_price = 10  

    def __init__(self, toppings):
        self.toppings = toppings

    # A class method using 'cls'
    @classmethod
    def update_base_price(cls, new_price):
        # cls points to the Pizza class, allowing us to change class attributes
        cls.base_price = new_price  

# Call the method directly on the class (no instance needed)
Pizza.update_base_price(12)
print(Pizza.base_price)  # Output: 12


12


`ctrl+shift+k` to delete a line in vscode

In [10]:
class Pizza:
    base_price = 10  

    def __init__(self, toppings):
        self.toppings = toppings

    def update_base_price(cls, new_price):
        cls.base_price = new_price  

Pizza.update_base_price(12)
print(Pizza.base_price)  # Output: 12


TypeError: Pizza.update_base_price() missing 1 required positional argument: 'new_price'

In [ ]:
class Pizza:
    base_price = 10

    def update_base_price(cls, new_price):
        cls.base_price = new_price

piz1=Pizza()
print(piz1.base_price)
piz2=Pizza()
print(piz2.base_price)

Pizza.update_base_price(Pizza, 12)
print(Pizza.base_price)  # 12
print(piz1.base_price)

piz2=Pizza()
print(piz2.base_price)

10
10
12
12
12


In [ ]:
class Pizza:
    base_price = 10  

    def __init__(self, toppings):
        self.toppings = toppings

    def update_base_price(new_price):
        base_price = new_price  

Pizza.update_base_price(12)
print(Pizza.base_price) 

# here we can call the function but it cant update the base price of the class


10


- https://chatgpt.com/share/6adc9382-c078-83e8-98ea-c44bf6838107

### 4. What is a staticmethod in python?

- A static method in Python is a method defined inside a class that does not depend on instance (self) or class (cls) data. It behaves exactly like a regular function, but it is placed inside a class namespace because it logically belongs there.

In [2]:
class MathOperations:
    
    @staticmethod
    def add(a, b):
        # Has no access to the class or instance state
        return a + b

# You call it directly on the Class without creating an object
result = MathOperations.add(5, 10)
print(result)  # Output: 15


15


In [ ]:
class MathOperations:
    
    
    def add(cls,a, b):
        return a + b

result = MathOperations.add(5, 10)
print(result)  # Output: 15


TypeError: MathOperations.add() missing 1 required positional argument: 'b'

### 5. where to use static method and class methods

- https://chatgpt.com/share/6adc94bd-4f30-83ee-b579-0cc9a77781a0

### 6. What are dataclasses in python?

The Python dataclasses module provides the @dataclass decorator to automatically generate boilerplate dunder methods like __init__, __repr__, and __eq__ based on PEP 526 type annotations. It simplifies creating classes whose primary purpose is to store data.

🛠️ Basic Usage

To create a data class, import the decorator from the dataclasses module and apply it to your class. Type hints are mandatory for field inclusion.

In [15]:
from dataclasses import dataclass

@dataclass
class Product:
    name: str
    price: float
    quantity: int = 0  # Default value

# Automatically generates __init__
item = Product(name="Laptop", price=999.99, quantity=2)

# Automatically generates a clean __repr__
print(item)  
# Output: Product(name='Laptop', price=999.99, quantity=2)

# Automatically generates __eq__ (compares values, not memory addresses)
item2 = Product(name="Laptop", price=999.99, quantity=2)
print(item == item2)  # Output: True


Product(name='Laptop', price=999.99, quantity=2)
True



⚙️ Core Customization & Key Features

- Mutable Default Values: You cannot use mutable defaults like [] or {} directly. Instead, use the field() function with a default_factory to ensure each instance gets its own independent copy.
- Immutability (frozen=True): Setting frozen=True makes instances read-only. Attempting to modify fields after initialization will raise a FrozenInstanceError.
- Post-Initialization (`__post_init__`): Used to compute derived fields or validate data right after the generated `__init__` finishes running.
- Memory Optimization (slots=True): Available in Python 3.10+, this builds the class using `__slots__`, reducing memory footprint and speeding up attribute access.

In [16]:
from dataclasses import dataclass, field

@dataclass(frozen=True, slots=True)
class User:
    username: str
    email: str
    # Handles mutable default cleanly
    roles: list[str] = field(default_factory=list) 
    # Excludes sensitive data from the string representation
    password_hash: str = field(repr=False, default="***") 

    def __post_init__(self):
        # Perform validation (Note: for frozen classes, adjustments require a workaround)
        if "@" not in self.email:
            raise ValueError("Invalid email address")



🧰 Built-in Serialization Utilities

The module provides helpers to easily convert your structured data class instances into primitive data structures:
- asdict(instance): Recursively converts a dataclass instance into a standard Python dictionary (perfect for JSON serialization).
- astuple(instance): Converts the instance into a tuple of its field values.